In [ ]:
import zmq
import numpy as np
import fastplotlib as fpl

In [ ]:
# ZMQ context and socket setup
context = zmq.Context()
socket = context.socket(zmq.SUB)
socket.connect("tcp://127.0.0.1:5555")
socket.setsockopt_string(zmq.SUBSCRIBE, "")
print("Listening...")

In [ ]:
# Create the figure
figure = fpl.Figure()

# Placeholder for the first data reception
is_first_data = True

# Set the dimension for reshaping
dimension = 2  # Default to 2 for [x, y] data; adjust as needed

In [ ]:
def get_buffer():
    """
    Retrieve the buffer from the socket.
    """
    try:
        b = socket.recv(zmq.NOBLOCK)  # Non-blocking receive
        return b
    except zmq.Again:
        return None

In [ ]:
def update_frame(p):
    """
    Update the frame using data received from the socket and reshape it based on the specified dimension.
    """
    global is_first_data

    buff = get_buffer()
    if buff is not None:
        # Deserialize the buffer into a NumPy array
        data = np.frombuffer(buff, dtype=np.float64)

        # Extract the frame number from the last index
        frame_num = int(data[-1])  # Last element is the frame number

        # Reshape the remaining data based on the specified dimension
        if dimension > 0:
            values = data[:-1].reshape(-1, dimension)  # Reshape all but the last element
        else:
            values = data[:-1]  # No reshaping if dimension <= 0

        if is_first_data:
            # Initialize the plot with the appropriate number of points
            n_points = (len(data) - 1) // dimension
            xs = np.linspace(-10, 10, n_points)
            ys = np.zeros_like(xs)
            figure[0, 0].add_line(data=np.column_stack((xs, ys)), name="wave")
            is_first_data = False

        # Update the line plot
        if dimension >= 2:  # Ensure at least [x, y] data is available for plotting
            p["wave"].data[:, :dimension] = values
            
        else:
            print(f"Received frame {frame_num}, but dimension {dimension} is insufficient for plotting.")

        # Update the plot title with the frame number
        p.name = f"frame: {frame_num}"

In [ ]:
# Add the animation update function
figure[0, 0].add_animations(update_frame)

figure.show()